> **Original course-project notebook (2026), kept for reference; superseded by `src/dfer/`.** Outputs are cleared: they contained DFEW frames (licence-restricted) and results that were selected on the fold's test set and affected by a frame-ordering bug. See the README for the corrected pipeline and results.

#Self-Supervised MAE-DFER for Emotion Recognition

In this notebook, we implement MAE-DFER, a self-supervised model fine-tuned for dynamic facial expression recognition (DFER) on the DFEW dataset. The model is based on VideoMAE pretrained on large video datasets, and we fine-tune it for the task of emotion classification.

We follow a two-stage approach for fine-tuning:

* Stage 1: Fine-tune only the classifier head of the model while keeping the encoder frozen.
* Stage 2: Unfreeze the last few layers of the encoder and fine-tune the entire model.

The goal of this setup is to understand how self-supervised pretraining can improve performance on the smaller DFEW dataset, which has a limited number of labeled samples.

##Imports and Setup

We begin by importing the necessary libraries:

* PyTorch: For model building and training.
* Transformers: To load the pretrained VideoMAE model.
* Matplotlib and tqdm: For visualization and progress tracking.
* YAML: For loading configuration files.

We also set up the necessary paths and configurations for the project.

In [ ]:
import os, sys, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from tqdm import tqdm
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

from transformers import VideoMAEForVideoClassification
import yaml

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

CONFIG_PATH = PROJECT_ROOT / "configs" / "paths.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DFEW_ROOT = config["dfew_root"]
SPLITS_ROOT = config["splits_root"]

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
LOG_DIR    = OUTPUT_DIR / "logs"
PLOT_DIR   = OUTPUT_DIR / "plots"

for d in [OUTPUT_DIR, CKPT_DIR, LOG_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DFEW_ROOT:", DFEW_ROOT)

##Reproducibility and Device Setup

In [ ]:
SEED = 42
FOLD = 1

NUM_FRAMES = 16
IMG_SIZE = 224
NUM_CLASSES = 7

BATCH_SIZE = 4
NUM_WORKERS = 4
PIN_MEMORY = True

EPOCHS_STAGE1 = 6
EPOCHS_STAGE2 = 14

PATIENCE = 5
MIN_DELTA = 1e-4

MODEL_ID = "MCG-NJU/videomae-base"
RUN_NAME = f"mae_dfer_fold{FOLD}_FINAL"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

##Data Loading and Transformation

We define the transformations for the train and validation datasets:

* Training transformations: Random resizing, horizontal flipping, and normalization to augment the dataset.
* Validation transformations: Resizing and normalization to prepare data for evaluation.

We then load the train and validation datasets using the DFEWVideoDataset class and create DataLoaders for batching the data.

In [ ]:
from dataset import DFEWVideoDataset

TRAIN_CSV = os.path.join(SPLITS_ROOT, "train(single-labeled)", f"set_{FOLD}.csv")
VAL_CSV   = os.path.join(SPLITS_ROOT, "test(single-labeled)",  f"set_{FOLD}.csv")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485,0.456,0.406),
                         std=(0.229,0.224,0.225))
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485,0.456,0.406),
                         std=(0.229,0.224,0.225))
])

train_ds = DFEWVideoDataset(DFEW_ROOT, TRAIN_CSV, train_transform, NUM_FRAMES)
val_ds   = DFEWVideoDataset(DFEW_ROOT, VAL_CSV,   val_transform,   NUM_FRAMES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=PIN_MEMORY)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

##Class Weights for Imbalance

To handle class imbalance in the dataset, we calculate class weights:

* Rarer classes are given higher weights to ensure they are treated fairly during training.
* The weights are calculated as the inverse of the class frequencies and normalized.

These class weights are used to adjust the loss function and improve model performance on underrepresented classes.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
counts = train_df["label"].value_counts().sort_index()

freq = counts.values.astype(np.float32)
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * len(weights)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

##Model Setup: MAE-DFER

In this section, we load the VideoMAE model that has been pretrained on a large video dataset. The model is adapted for emotion classification by replacing its final layer with a classifier suited to the 7 classes in the DFEW dataset.

We use the pretrained weights from the model and move it to the appropriate device (GPU or CPU) for training.

In [ ]:
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.amp.GradScaler("cuda") if device=="cuda" else None

##Training and Validation Loops

In [ ]:
def train_stage(num_epochs, stage_name, optimizer, scheduler, start_epoch_offset=0):

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "val_uar": [],
        "lr": [],
        "epoch_time_min": []
    }

    best_acc = -1.0
    best_uar = -1.0
    no_improve_uar = 0

    best_acc_path = CKPT_DIR / f"{RUN_NAME}_{stage_name}_best_acc.pth"
    best_uar_path = CKPT_DIR / f"{RUN_NAME}_{stage_name}_best_uar.pth"

    for epoch in range(num_epochs):
        epoch_start = time.time()

        # ---------------- TRAIN ----------------
        model.train()
        train_losses, train_preds, train_targets = [], [], []

        pbar = tqdm(train_loader, desc=f"{stage_name} Epoch {epoch+1}/{num_epochs}", leave=True)

        for videos, labels in pbar:
            if videos.shape[1] == 3:
                videos = videos.permute(0,2,1,3,4)

            videos = videos.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            if device == "cuda":
                with torch.amp.autocast("cuda"):
                    logits = model(videos).logits
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(videos).logits
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

            train_losses.append(loss.item())
            preds = logits.argmax(1).detach().cpu().numpy()
            train_preds.extend(preds)
            train_targets.extend(labels.detach().cpu().numpy())

            pbar.set_postfix({
                "loss": float(np.mean(train_losses)),
                "acc": float(accuracy_score(train_targets, train_preds)),
                "lr": float(optimizer.param_groups[0]["lr"])
            })

        train_loss = float(np.mean(train_losses))
        train_acc  = float(accuracy_score(train_targets, train_preds))

        # ---------------- VAL ----------------
        model.eval()
        val_losses, val_preds, val_targets = [], [], []

        with torch.no_grad():
            for videos, labels in val_loader:
                if videos.shape[1] == 3:
                    videos = videos.permute(0,2,1,3,4)

                videos = videos.to(device)
                labels = labels.to(device)

                logits = model(videos).logits
                loss = criterion(logits, labels)

                val_losses.append(loss.item())
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_targets.extend(labels.cpu().numpy())

        val_loss = float(np.mean(val_losses))
        val_acc  = float(accuracy_score(val_targets, val_preds))
        val_uar  = float(recall_score(val_targets, val_preds, average="macro"))

        scheduler.step()
        lr_now = float(optimizer.param_groups[0]["lr"])
        epoch_time = (time.time() - epoch_start) / 60.0

        global_epoch = start_epoch_offset + epoch + 1

        print(f"\n[{stage_name}] Epoch {global_epoch}")
        print(f"Train Loss {train_loss:.4f} | Acc {train_acc:.4f}")
        print(f"Val   Loss {val_loss:.4f} | Acc {val_acc:.4f} | UAR {val_uar:.4f}")
        print(f"LR {lr_now:.6f} | Time {epoch_time:.2f} min")

        history["epoch"].append(global_epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_uar"].append(val_uar)
        history["lr"].append(lr_now)
        history["epoch_time_min"].append(epoch_time)

        # ---------------- CHECKPOINTS ----------------
        if val_acc > best_acc + MIN_DELTA:
            best_acc = val_acc
            torch.save(model.state_dict(), best_acc_path)
            print("New BEST-ACC saved:", best_acc_path.name)

        if val_uar > best_uar + MIN_DELTA:
            best_uar = val_uar
            no_improve_uar = 0
            torch.save(model.state_dict(), best_uar_path)
            print("New BEST-UAR saved:", best_uar_path.name)
        else:
            no_improve_uar += 1
            print(f"No UAR improvement {no_improve_uar}/{PATIENCE}")

        if no_improve_uar >= PATIENCE:
            print("Early stopping (UAR).")
            break

    return history, best_acc_path, best_uar_path

##Stage 1: Fine-Tuning the Classifier

In the first stage, we fine-tune only the classifier head of the VideoMAE model. The encoder layers are kept frozen to retain the general video representations learned from the pretraining dataset.

We use:

* Learning Rate: 1e-3 for the classifier.
* Weight Decay: Regularization for preventing overfitting.
* Cosine Annealing Scheduler: Adjusts the learning rate during training to help the model converge.

The purpose of this stage is to adapt the classifier to the DFEW task while preserving the general representations learned from large-scale pretraining.

In [ ]:
for p in model.videomae.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

optimizer1 = torch.optim.AdamW(model.classifier.parameters(),
                               lr=1e-3, weight_decay=0.01)

scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer1, T_max=EPOCHS_STAGE1, eta_min=1e-6)

hist1, best_acc1, best_uar1 = train_stage(EPOCHS_STAGE1,
                                          "stage1_head",
                                          optimizer1,
                                          scheduler1,
                                          start_epoch_offset=0)

##Stage 2: Fine-Tuning the Encoder

In the second stage, we unfreeze the last few layers of the encoder and fine-tune the entire model. This allows the model to adapt to the specific emotion classification task by updating the encoder’s parameters.

We use a smaller learning rate for the encoder layers and the classifier:

* Learning Rate: 3e-5 for the encoder layers and 5e-4 for the classifier.
* Cosine Annealing Scheduler: Adjusts the learning rate during training.

The purpose of this stage is to fine-tune the model further by updating the encoder's layers and making the model more task-specific.

In [ ]:
for p in model.videomae.parameters():
    p.requires_grad = False

for block in model.videomae.encoder.layer[-2:]:
    for p in block.parameters():
        p.requires_grad = True

for p in model.videomae.layernorm.parameters():
    p.requires_grad = True

for p in model.classifier.parameters():
    p.requires_grad = True

optimizer2 = torch.optim.AdamW([
    {"params": model.videomae.encoder.layer[-2:].parameters(), "lr": 3e-5},
    {"params": model.videomae.layernorm.parameters(), "lr": 3e-5},
    {"params": model.classifier.parameters(), "lr": 5e-4},
], weight_decay=0.05)

scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=EPOCHS_STAGE2, eta_min=1e-6)

hist2, best_acc2, best_uar2 = train_stage(EPOCHS_STAGE2,
                                          "stage2_partial",
                                          optimizer2,
                                          scheduler2,
                                          start_epoch_offset=len(hist1["epoch"]))

##Plots (Acc , Loss , UAR)

In [ ]:
hist = pd.DataFrame({k: hist1[k] + hist2[k] for k in hist1})

# LOSS
plt.figure()
plt.plot(hist["epoch"], hist["train_loss"], label="train")
plt.plot(hist["epoch"], hist["val_loss"], label="val")
plt.legend(); plt.grid(); plt.title(RUN_NAME+" Loss")
plt.show()

# ACC
plt.figure()
plt.plot(hist["epoch"], hist["train_acc"], label="train")
plt.plot(hist["epoch"], hist["val_acc"], label="val")
plt.legend(); plt.grid(); plt.title(RUN_NAME+" Accuracy")
plt.show()

# UAR
plt.figure()
plt.plot(hist["epoch"], hist["val_uar"], label="val_uar")
plt.legend(); plt.grid(); plt.title(RUN_NAME+" UAR")
plt.show()

##Confusion Matrices (Acc , UAR)

In [ ]:
def eval_cm(weights_path, title):

    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.to(device)
    model.eval()

    all_preds, all_targets = [], []

    with torch.no_grad():
        for videos, labels in val_loader:
            if videos.shape[1] == 3:
                videos = videos.permute(0,2,1,3,4)
            videos = videos.to(device)
            logits = model(videos).logits
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_targets.extend(labels.numpy())

    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(6,6))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.show()

    print("Acc:", accuracy_score(all_targets, all_preds))
    print("UAR:", recall_score(all_targets, all_preds, average="macro"))

eval_cm(best_acc2, RUN_NAME+" BEST ACC")
eval_cm(best_uar2, RUN_NAME+" BEST UAR")